# Извлечение именованных сущностей (NER) из юридических документов с помощью YandexGPT

**Курс:** LLM Driven Development · **Тема:** промпт-инжиниринг + интеграция облачной LLM по API.

Пайплайн автоматически извлекает из договоров четыре типа сущностей и отдаёт их структурированным JSON:

1. **Реквизиты сторон** — название организации, ИНН, КПП, роль;
2. **Даты** — дата подписания и срок действия;
3. **Суммы** — числовое значение + валюта;
4. **Сроки обязательств** — в днях/месяцах или конкретными датами.

**Что внутри:** подготовка текста (truncation/chunking) → система промптов (system + few-shot) →
обращение к API YandexGPT с низкой `temperature` → устойчивый парсер JSON (чистка markdown + починка
сломанных скобок) → тесты на edge cases (отсутствие данных, шум OCR) → отчёт и сравнение с Natasha/spaCy.

> ⚠️ **Ключи API в код не зашиты** — берутся из переменных окружения или запрашиваются через `getpass`.
> По умолчанию включён `USE_MOCK=True` (демо без ключа/сети). Для реального прогона поставьте `USE_MOCK=False`.

## 0. Установка зависимостей

In [1]:
# Ставим зависимости. requests — для обращения к REST API YandexGPT.
# (в Colab выполните эту ячейку один раз)
!pip install -q requests

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 1. Аутентификация (без ключей в коде)

In [2]:
# ============================================================
#  АУТЕНТИФИКАЦИЯ.  ВАЖНО: ключи НЕ хранятся в коде!
# ============================================================
# Ключи берутся из переменных окружения, а если их нет —
# запрашиваются интерактивно (getpass прячет ввод в Colab).
#
# Как получить (по инструкции к уроку):
#   1. Создать каталог в Yandex Cloud -> скопировать folder_id.
#   2. Создать сервисный аккаунт, выдать роль ai.languageModels.user.
#   3. Создать API-ключ этого сервисного аккаунта.
# ------------------------------------------------------------
import os
from getpass import getpass

YANDEX_API_KEY = os.environ.get("YANDEX_API_KEY")
if not YANDEX_API_KEY:
    YANDEX_API_KEY = getpass("Введите Yandex Cloud API-ключ (ввод скрыт): ").strip()

YANDEX_FOLDER_ID = os.environ.get("YANDEX_FOLDER_ID")
if not YANDEX_FOLDER_ID:
    YANDEX_FOLDER_ID = input("Введите folder_id каталога: ").strip()

# Модель: 'yandexgpt' (Pro) или 'yandexgpt-lite' (дешевле/быстрее).
MODEL_NAME = os.environ.get("YANDEX_MODEL", "yandexgpt")
MODEL_URI = f"gpt://{YANDEX_FOLDER_ID}/{MODEL_NAME}/latest"

# ------------------------------------------------------------
#  РЕЖИМ ДЕМО.  USE_MOCK=True -> ноутбук работает БЕЗ ключа и
#  сети: вместо реального API отдаются заранее записанные
#  ответы модели (см. ячейку MOCK_RESPONSES). Это позволяет
#  прогнать весь пайплайн и проверить парсер/обработку ошибок.
#
#  Перед сдачей с реальными данными:  поставьте USE_MOCK = False
# ------------------------------------------------------------
USE_MOCK = True

# Использовать ли режим СТРУКТУРИРОВАННОГО ВЫВОДА (JSON Schema).
# При True и USE_MOCK=False запрос уходит с полем json_schema — API
# гарантирует валидный JSON нужной структуры (нужна свежая модель, напр.
# yandexgpt/rc или yandexgpt/latest). При USE_MOCK=True флаг не влияет.
USE_STRUCTURED_OUTPUT = False

print("folder_id получен      :", bool(YANDEX_FOLDER_ID))
print("api-key получен (длина):", len(YANDEX_API_KEY) if YANDEX_API_KEY else 0)
print("modelUri               :", f"gpt://<folder_id>/{MODEL_NAME}/latest")
print("USE_MOCK               :", USE_MOCK)
print("USE_STRUCTURED_OUTPUT  :", USE_STRUCTURED_OUTPUT)

folder_id получен      : True
api-key получен (длина): 28
modelUri               : gpt://<folder_id>/yandexgpt/latest
USE_MOCK               : True
USE_STRUCTURED_OUTPUT  : False


## 2. Тестовый датасет юридических документов

7 синтетических документов (данные выдуманы). Среди них — граничные случаи:
`doc_04` (NDA) и `doc_05` (доверенность) без сумм, `doc_06` — с имитацией плохого OCR.

In [3]:
# -*- coding: utf-8 -*-
"""Синтетический датасет юридических документов (7 шт.) для тестирования NER.

Данные полностью выдуманы. ИНН/КПП сгенерированы в правильном формате,
но не принадлежат реальным организациям. Специально включены edge cases:
  - doc_04 (NDA)         -> нет сумм и явных денежных сроков
  - doc_05 (Доверенность) -> нет сумм
  - doc_06 (OCR-шум)     -> имитация плохого распознавания скана (опечатки)
"""

DOCUMENTS = [
    {
        "id": "doc_01_supply",
        "title": "Договор поставки",
        "text": """ДОГОВОР ПОСТАВКИ № ПС-2026/114

г. Москва                                                   12 марта 2026 г.

Общество с ограниченной ответственностью «Ромашка», ИНН 7701234567,
КПП 770101001, именуемое в дальнейшем «Поставщик», в лице генерального
директора Иванова Ивана Ивановича, с одной стороны, и
Акционерное общество «Василёк», ИНН 7809876543, КПП 780101001,
именуемое в дальнейшем «Покупатель», в лице директора Петровой Марии
Сергеевны, с другой стороны, заключили настоящий договор о нижеследующем.

1. Поставщик обязуется поставить, а Покупатель принять и оплатить
   офисную мебель на общую сумму 1 500 000 (один миллион пятьсот тысяч)
   рублей 00 копеек, в том числе НДС 20%.
2. Поставка осуществляется в течение 30 (тридцати) календарных дней с
   момента подписания договора.
3. Покупатель производит оплату в течение 10 рабочих дней после получения
   товара.
4. Настоящий договор вступает в силу с момента подписания и действует до
   31 декабря 2026 года.
""",
    },
    {
        "id": "doc_02_services",
        "title": "Договор оказания услуг",
        "text": """ДОГОВОР ВОЗМЕЗДНОГО ОКАЗАНИЯ УСЛУГ № У-77/2026

г. Санкт-Петербург                                          05.02.2026

Индивидуальный предприниматель Сидоров Алексей Петрович,
ИНН 780512345678, именуемый в дальнейшем «Исполнитель», и
ООО «ТехноСервис», ИНН 5024112233, КПП 502401001, именуемое
«Заказчик», заключили договор о нижеследующем.

Исполнитель обязуется оказать услуги по настройке ИТ-инфраструктуры.
Стоимость услуг составляет 340 000 рублей (НДС не облагается).
Аванс в размере 50% выплачивается в срок не позднее 3 (трёх) банковских
дней с даты подписания. Итоговый расчёт — в течение 15 календарных дней
после подписания акта. Срок оказания услуг — 45 календарных дней.
Договор действует до 05 августа 2026 года.
""",
    },
    {
        "id": "doc_03_lease",
        "title": "Договор аренды",
        "text": """ДОГОВОР АРЕНДЫ НЕЖИЛОГО ПОМЕЩЕНИЯ № А-2026-09

г. Казань                                                   «20» января 2026 года

Арендодатель: ООО «Недвижимость Плюс», ИНН 1655098765, КПП 165501001.
Арендатор: ЗАО «Логистик Групп», ИНН 7743001122, КПП 774301001.

Арендодатель передаёт Арендатору во временное пользование складское
помещение площадью 800 кв. м. Размер ежемесячной арендной платы
составляет 250 000 (двести пятьдесят тысяч) рублей, включая НДС.
Обеспечительный платёж — 500 000 рублей, вносится в течение 5 рабочих
дней с даты подписания. Арендная плата вносится ежемесячно не позднее
10-го числа. Срок аренды — 11 месяцев с 01 февраля 2026 года по
31 декабря 2026 года.
""",
    },
    {
        "id": "doc_04_nda",
        "title": "NDA (без сумм — edge case)",
        "text": """СОГЛАШЕНИЕ О НЕРАЗГЛАШЕНИИ КОНФИДЕНЦИАЛЬНОЙ ИНФОРМАЦИИ № NDA-14

г. Москва                                                   03 апреля 2026 г.

ООО «Альфа Разработка», ИНН 7728555666, КПП 772801001 («Сторона 1»), и
ООО «Бета Консалтинг», ИНН 7702998877, КПП 770201001 («Сторона 2»),
договорились о следующем.

Стороны обязуются не разглашать конфиденциальную информацию, полученную
в рамках сотрудничества. Обязательства по неразглашению действуют в
течение 3 (трёх) лет с момента прекращения сотрудничества. Настоящее
соглашение не предусматривает денежных расчётов между Сторонами.
Соглашение вступает в силу с даты подписания и действует бессрочно в
части обязательств о конфиденциальности.
""",
    },
    {
        "id": "doc_05_poa",
        "title": "Доверенность (без сумм — edge case)",
        "text": """ДОВЕРЕННОСТЬ № 27

г. Новосибирск                                              15 мая 2026 года

Общество с ограниченной ответственностью «Сибирь Трейд», ИНН 5406123456,
КПП 540601001, в лице директора Кузнецова Дмитрия Олеговича, настоящей
доверенностью уполномочивает Смирнова Олега Николаевича представлять
интересы Общества в органах ФНС России, подписывать и подавать
отчётность, получать документы.

Доверенность выдана сроком на 1 (один) год без права передоверия и
действует до 15 мая 2027 года.
""",
    },
    {
        "id": "doc_06_ocr_noise",
        "title": "Договор с OCR-опечатками (шум — edge case)",
        "text": """ДOГOВOР ПOДРЯДA № ПД-2О26/O8

г. Eкатеринбург                                            «18» феараля 2О26 r.

OOO «CтрoйМaстер», ИHН 66581Ol234, KПП 665801OO1, именуемoе «Пoдрядчик»,
и AO «Девелoпмент Урал», ИНН 66З4009988, КПП 6634OlOO1, именуемoе
«3aказчик», закпючили нaстоящий дoгoвoр.

Пoдрядчик oбязуется вьполнить ремoнтнье рабoты. Cтoимость работ пo
дoгoвору сoставляет 2 8ОO OОO (два миллиoна вoсемьсoт тысяч) рyблей,
в тoм числе НДC. Aвaнс 30% в течeние 7 рабoчих дней. Cрок вьполнения
рабoт — 6О календaрных дней с мoмента пoдписания. Дoговoр дeйствует
дo 18 aвгуста 2026 гoда.
""",
    },
    {
        "id": "doc_07_loan",
        "title": "Договор займа",
        "text": """ДОГОВОР ЗАЙМА № З-2026/45

г. Нижний Новгород                                          10 июня 2026 г.

Займодавец: ООО «Финанс Капитал», ИНН 5260112233, КПП 526001001.
Заёмщик: ООО «Производство НН», ИНН 5262556677, КПП 526201001.

Займодавец передаёт Заёмщику денежные средства в размере
5 000 000 (пять миллионов) рублей под 12% годовых. Заёмщик обязуется
вернуть сумму займа и проценты в срок до 10 декабря 2026 года.
Проценты уплачиваются ежемесячно не позднее 25-го числа каждого месяца.
Договор вступает в силу с момента передачи денежных средств и действует
до полного исполнения обязательств.
""",
    },
]

print('Документов в датасете:', len(DOCUMENTS))

Документов в датасете: 7


## 3. Базовая подготовка текстов (длина, truncation, chunking)

In [4]:
# ============================================================
#  БАЗОВАЯ ПОДГОТОВКА ТЕКСТОВ
# ============================================================
# У LLM ограничено контекстное окно. Оцениваем длину и, если
# документ слишком большой, либо обрезаем (truncation), либо
# режем на смысловые части (chunking по абзацам).
# Грубая оценка: для русского ~1 токен ≈ 2 символа.
# ------------------------------------------------------------

MAX_INPUT_CHARS = 12000   # порог, выше которого включаем обрезку/чанкинг


def estimate_tokens(text: str) -> int:
    """Грубая оценка числа токенов (для контроля лимитов)."""
    return len(text) // 2


def truncate(text: str, max_chars: int = MAX_INPUT_CHARS) -> str:
    """Простая обрезка слишком длинного текста."""
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...[текст обрезан]"


def chunk_by_paragraphs(text: str, max_chars: int = MAX_INPUT_CHARS):
    """Разбиение на смысловые части по абзацам (пустая строка = граница)."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) + 2 <= max_chars:
            current = (current + "\n\n" + p).strip()
        else:
            if current:
                chunks.append(current)
            current = p
    if current:
        chunks.append(current)
    return chunks or [text[:max_chars]]


print("Проверка длины документов:")
print(f"{'id':<20}{'символов':>10}{'~токенов':>10}{'обрезка?':>12}")
for d in DOCUMENTS:
    n = len(d["text"])
    print(f"{d['id']:<20}{n:>10}{estimate_tokens(d['text']):>10}"
          f"{('да' if n > MAX_INPUT_CHARS else 'нет'):>12}")

Проверка длины документов:
id                    символов  ~токенов    обрезка?
doc_01_supply              975       487         нет
doc_02_services            728       364         нет
doc_03_lease               688       344         нет
doc_04_nda                 694       347         нет
doc_05_poa                 504       252         нет
doc_06_ocr_noise           590       295         нет
doc_07_loan                602       301         нет


## 4. Промпты (system + пользовательский с few-shot)

In [5]:
# ============================================================
#  PROMPT ENGINEERING
# ============================================================

# --- Системный промпт: задаём строгую роль ---
SYSTEM_PROMPT = (
    "Ты — строгий и точный AI-помощник юриста. "
    "Твоя задача — извлекать факты из текста договоров без искажений и "
    "без домысливания. Если какого-то поля в тексте нет — верни для него "
    "null (для списков — пустой массив). Никогда не выдумывай значения. "
    "Отвечай ТОЛЬКО валидным JSON-объектом, без пояснений и без markdown."
)

# --- Целевая JSON-схема (описываем модели ожидаемую структуру) ---
JSON_SCHEMA_HINT = """{
  "номер_договора": "строка или null",
  "стороны": [
    {"роль": "строка (напр. Поставщик/Покупатель)", "название": "строка",
     "ИНН": "строка или null", "КПП": "строка или null"}
  ],
  "даты": {"дата_подписания": "строка или null", "срок_действия": "строка или null"},
  "суммы": [
    {"значение": "число", "валюта": "код валюты, напр. RUB", "описание": "за что"}
  ],
  "сроки_обязательств": [
    {"описание": "суть обязательства", "срок": "в днях/месяцах или дата"}
  ]
}"""

# --- Few-shot: показываем ОДИН пример вход->выход ---
FEWSHOT_INPUT = (
    "ДОГОВОР № Д-1. ООО «Пример», ИНН 7700000001, КПП 770000001 "
    "(«Исполнитель») и АО «Клиент», ИНН 7800000002, КПП 780000001 "
    "(«Заказчик»). Стоимость 100 000 рублей. Оплата в течение 5 дней. "
    "Подписан 01.01.2026, действует до 01.07.2026."
)

FEWSHOT_OUTPUT = """{
  "номер_договора": "Д-1",
  "стороны": [
    {"роль": "Исполнитель", "название": "ООО «Пример»", "ИНН": "7700000001", "КПП": "770000001"},
    {"роль": "Заказчик", "название": "АО «Клиент»", "ИНН": "7800000002", "КПП": "780000001"}
  ],
  "даты": {"дата_подписания": "01.01.2026", "срок_действия": "до 01.07.2026"},
  "суммы": [{"значение": 100000, "валюта": "RUB", "описание": "стоимость услуг"}],
  "сроки_обязательств": [{"описание": "оплата", "срок": "5 дней"}]
}"""


def build_user_prompt(document_text: str) -> str:
    """Собираем пользовательский промпт: инструкция + схема + few-shot + документ."""
    return (
        "Извлеки из текста договора именованные сущности и верни JSON строго "
        "по следующей структуре:\n"
        f"{JSON_SCHEMA_HINT}\n\n"
        "Извлекай ровно эти типы данных:\n"
        "1) Реквизиты сторон: название организации, ИНН, КПП, роль стороны.\n"
        "2) Даты: дата подписания и срок действия договора.\n"
        "3) Суммы: числовое значение и валюта (RUB/USD/EUR).\n"
        "4) Сроки обязательств: в днях/месяцах или конкретными датами.\n\n"
        "Правила: не выдумывай данные; отсутствующее поле = null; "
        "ответ — только JSON без markdown.\n\n"
        "=== ПРИМЕР ===\n"
        f"Текст: {FEWSHOT_INPUT}\n"
        f"JSON: {FEWSHOT_OUTPUT}\n\n"
        "=== ЗАДАНИЕ ===\n"
        f"Текст: {document_text}\n"
        "JSON:"
    )


print("Системный промпт:\n", SYSTEM_PROMPT)
print("\nПример пользовательского промпта (фрагмент):\n")
print(build_user_prompt("<ТЕКСТ ДОГОВОРА>")[:600], "...")

Системный промпт:
 Ты — строгий и точный AI-помощник юриста. Твоя задача — извлекать факты из текста договоров без искажений и без домысливания. Если какого-то поля в тексте нет — верни для него null (для списков — пустой массив). Никогда не выдумывай значения. Отвечай ТОЛЬКО валидным JSON-объектом, без пояснений и без markdown.

Пример пользовательского промпта (фрагмент):

Извлеки из текста договора именованные сущности и верни JSON строго по следующей структуре:
{
  "номер_договора": "строка или null",
  "стороны": [
    {"роль": "строка (напр. Поставщик/Покупатель)", "название": "строка",
     "ИНН": "строка или null", "КПП": "строка или null"}
  ],
  "даты": {"дата_подписания": "строка или null", "срок_действия": "строка или null"},
  "суммы": [
    {"значение": "число", "валюта": "код валюты, напр. RUB", "описание": "за что"}
  ],
  "сроки_обязательств": [
    {"описание": "суть обязательства", "срок": "в днях/месяцах или дата"}
  ]
}

Извлекай ровно эти тип ...


## 5. Обращение к API YandexGPT

Синхронный REST-запрос к `.../foundationModels/v1/completion`. `temperature=0.2` — низкая,
чтобы минимизировать креативность и галлюцинации. Есть повтор при сетевых сбоях.

In [6]:
# ============================================================
#  ОБРАЩЕНИЕ К API YandexGPT (REST)
# ============================================================
import time
import requests

COMPLETION_URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"


def call_yandexgpt_real(system_prompt: str, user_prompt: str,
                        temperature: float = 0.2, max_tokens: int = 2000,
                        retries: int = 3) -> str:
    """Синхронный запрос к YandexGPT. Возвращает текст ответа модели.

    Низкая temperature (0.1–0.3) минимизирует креативность и галлюцинации,
    что критично для задачи извлечения фактов.
    """
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {YANDEX_API_KEY}",
        "x-folder-id": YANDEX_FOLDER_ID,
    }
    payload = {
        "modelUri": MODEL_URI,
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": str(max_tokens),
        },
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt},
        ],
    }
    last_err = None
    for attempt in range(retries):
        try:
            resp = requests.post(COMPLETION_URL, headers=headers,
                                 json=payload, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            return data["result"]["alternatives"][0]["message"]["text"]
        except Exception as e:  # сеть/квоты/5xx — повторяем с задержкой
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"API недоступно после {retries} попыток: {last_err}")

## 5b. Структурированный вывод по JSON Schema (надёжная альтернатива)

Более надёжный способ, чем «просить JSON в промпте»: передать API строгую **JSON-схему** через
поле `json_schema`. Тогда модель обязана вернуть валидный JSON нужной структуры — авто-починка
скобок почти не нужна. Требуется свежая модель (`yandexgpt/rc` или `yandexgpt/latest`).
Включается флагом `USE_STRUCTURED_OUTPUT = True` (действует при `USE_MOCK = False`).

> Есть и OpenAI-совместимый вариант: эндпоинт `https://ai.api.cloud.yandex.net/v1/chat/completions`
> с `response_format={"type": "json_schema", "json_schema": {...}}`.

In [7]:
# ============================================================
#  СТРУКТУРИРОВАННЫЙ ВЫВОД ПО JSON SCHEMA (надёжный способ)
# ============================================================
# Вместо надежды на промпт можно передать API строгую JSON-схему —
# тогда модель обязана вернуть валидный JSON нужной структуры,
# и обвязка с починкой скобок почти не нужна.
# ------------------------------------------------------------

NER_JSON_SCHEMA = {
    "schema": {
        "type": "object",
        "properties": {
            "номер_договора": {"type": ["string", "null"]},
            "стороны": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "роль": {"type": "string"},
                        "название": {"type": "string"},
                        "ИНН": {"type": ["string", "null"]},
                        "КПП": {"type": ["string", "null"]},
                    },
                    "required": ["роль", "название"],
                },
            },
            "даты": {
                "type": "object",
                "properties": {
                    "дата_подписания": {"type": ["string", "null"]},
                    "срок_действия": {"type": ["string", "null"]},
                },
            },
            "суммы": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "значение": {"type": "number"},
                        "валюта": {"type": "string"},
                        "описание": {"type": "string"},
                    },
                    "required": ["значение", "валюта"],
                },
            },
            "сроки_обязательств": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "описание": {"type": "string"},
                        "срок": {"type": "string"},
                    },
                },
            },
        },
        "required": ["стороны", "даты", "суммы", "сроки_обязательств"],
    }
}


def call_yandexgpt_structured(system_prompt: str, user_prompt: str,
                              temperature: float = 0.2, max_tokens: int = 2000,
                              retries: int = 3) -> str:
    """Запрос с гарантией структуры: поле json_schema в теле запроса.
    Требует свежую модель (yandexgpt/rc или yandexgpt/latest)."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {YANDEX_API_KEY}",
        "x-folder-id": YANDEX_FOLDER_ID,
    }
    payload = {
        "modelUri": MODEL_URI,
        "completionOptions": {"stream": False, "temperature": temperature,
                              "maxTokens": str(max_tokens)},
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt},
        ],
        "json_schema": NER_JSON_SCHEMA,   # <-- ключевое отличие
    }
    last_err = None
    for attempt in range(retries):
        try:
            resp = requests.post(COMPLETION_URL, headers=headers,
                                 json=payload, timeout=60)
            resp.raise_for_status()
            return resp.json()["result"]["alternatives"][0]["message"]["text"]
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"API недоступно после {retries} попыток: {last_err}")


print("JSON-схема для structured output готова. Полей верхнего уровня:",
      len(NER_JSON_SCHEMA["schema"]["properties"]))

JSON-схема для structured output готова. Полей верхнего уровня: 5


## 6. Парсер ответа LLM

Убирает markdown-обёртку ```` ```json ````, вырезает JSON-объект, при поломке
пытается починить (висячие запятые, незакрытые скобки) и распарсить повторно.

In [8]:
# ============================================================
#  ПАРСЕР ОТВЕТА LLM
# ============================================================
import re
import json


def _strip_markdown(text: str) -> str:
    """Убираем markdown-обёртку ```json ... ``` и лишний текст вокруг."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text).strip()
    # оставляем подстроку от первой { до последней }
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]
    return text


def _try_repair(text: str) -> str:
    """Чиним частые поломки: висячие запятые и незакрытые скобки."""
    # висячие запятые перед } или ]
    text = re.sub(r",\s*([}\]])", r"\1", text)
    # добираем недостающие закрывающие скобки
    opens_c, closes_c = text.count("{"), text.count("}")
    opens_b, closes_b = text.count("["), text.count("]")
    text += "]" * max(0, opens_b - closes_b)
    text += "}" * max(0, opens_c - closes_c)
    return text


def parse_llm_json(raw_text: str):
    """Возвращает (dict | None, error | None).

    1) чистим markdown; 2) пробуем json.loads;
    3) при ошибке — пробуем починить и распарсить ещё раз.
    """
    cleaned = _strip_markdown(raw_text)
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError as e1:
        repaired = _try_repair(cleaned)
        try:
            result = json.loads(repaired)
            return result, f"JSON был повреждён и восстановлен (исходно: {e1})"
        except json.JSONDecodeError as e2:
            return None, f"Не удалось распарсить JSON: {e2}"


# мини-тест парсера на «грязном» ответе
_demo = "```json\n{\"a\": 1, \"b\": [1, 2,],}\n```"
print("Тест парсера на грязном ответе:", parse_llm_json(_demo))

Тест парсера на грязном ответе: ({'a': 1, 'b': [1, 2]}, 'JSON был повреждён и восстановлен (исходно: Expecting value: line 1 column 21 (char 20))')


## 7. Демо-ответы модели (только для `USE_MOCK=True`)

In [9]:
# ============================================================
#  ДЕМО-ОТВЕТЫ МОДЕЛИ (используются только при USE_MOCK=True)
# ============================================================
# Это заранее записанные "сырые" ответы LLM для каждого документа.
# Они специально неидеальны, чтобы показать работу парсера:
#   - doc_01: ответ обёрнут в markdown ```json ... ```
#   - doc_04/doc_05: пустые суммы (модель НЕ галлюцинирует) — верное поведение
#   - doc_06: сломанный JSON (незакрытые скобки) + OCR-ошибка в КПП
# ------------------------------------------------------------

MOCK_RESPONSES = {

"doc_01_supply": '''```json
{
  "номер_договора": "ПС-2026/114",
  "стороны": [
    {"роль": "Поставщик", "название": "ООО «Ромашка»", "ИНН": "7701234567", "КПП": "770101001"},
    {"роль": "Покупатель", "название": "АО «Василёк»", "ИНН": "7809876543", "КПП": "780101001"}
  ],
  "даты": {"дата_подписания": "12 марта 2026 г.", "срок_действия": "до 31 декабря 2026 года"},
  "суммы": [{"значение": 1500000, "валюта": "RUB", "описание": "поставка офисной мебели, в т.ч. НДС 20%"}],
  "сроки_обязательств": [
    {"описание": "поставка товара", "срок": "30 календарных дней"},
    {"описание": "оплата после получения товара", "срок": "10 рабочих дней"}
  ]
}
```''',

"doc_02_services": '''{
  "номер_договора": "У-77/2026",
  "стороны": [
    {"роль": "Исполнитель", "название": "ИП Сидоров Алексей Петрович", "ИНН": "780512345678", "КПП": null},
    {"роль": "Заказчик", "название": "ООО «ТехноСервис»", "ИНН": "5024112233", "КПП": "502401001"}
  ],
  "даты": {"дата_подписания": "05.02.2026", "срок_действия": "до 05 августа 2026 года"},
  "суммы": [{"значение": 340000, "валюта": "RUB", "описание": "стоимость услуг, НДС не облагается"}],
  "сроки_обязательств": [
    {"описание": "аванс 50%", "срок": "3 банковских дня"},
    {"описание": "итоговый расчёт после акта", "срок": "15 календарных дней"},
    {"описание": "оказание услуг", "срок": "45 календарных дней"}
  ]
}''',

"doc_03_lease": '''{
  "номер_договора": "А-2026-09",
  "стороны": [
    {"роль": "Арендодатель", "название": "ООО «Недвижимость Плюс»", "ИНН": "1655098765", "КПП": "165501001"},
    {"роль": "Арендатор", "название": "ЗАО «Логистик Групп»", "ИНН": "7743001122", "КПП": "774301001"}
  ],
  "даты": {"дата_подписания": "20 января 2026 года", "срок_действия": "с 01.02.2026 по 31.12.2026"},
  "суммы": [
    {"значение": 250000, "валюта": "RUB", "описание": "ежемесячная арендная плата, включая НДС"},
    {"значение": 500000, "валюта": "RUB", "описание": "обеспечительный платёж"}
  ],
  "сроки_обязательств": [
    {"описание": "обеспечительный платёж", "срок": "5 рабочих дней"},
    {"описание": "внесение арендной платы", "срок": "не позднее 10-го числа ежемесячно"},
    {"описание": "срок аренды", "срок": "11 месяцев"}
  ]
}''',

"doc_04_nda": '''{
  "номер_договора": "NDA-14",
  "стороны": [
    {"роль": "Сторона 1", "название": "ООО «Альфа Разработка»", "ИНН": "7728555666", "КПП": "772801001"},
    {"роль": "Сторона 2", "название": "ООО «Бета Консалтинг»", "ИНН": "7702998877", "КПП": "770201001"}
  ],
  "даты": {"дата_подписания": "03 апреля 2026 г.", "срок_действия": "бессрочно в части конфиденциальности"},
  "суммы": [],
  "сроки_обязательств": [
    {"описание": "неразглашение после прекращения сотрудничества", "срок": "3 года"}
  ]
}''',

"doc_05_poa": '''{
  "номер_договора": "27",
  "стороны": [
    {"роль": "Доверитель", "название": "ООО «Сибирь Трейд»", "ИНН": "5406123456", "КПП": "540601001"}
  ],
  "даты": {"дата_подписания": "15 мая 2026 года", "срок_действия": "до 15 мая 2027 года"},
  "суммы": [],
  "сроки_обязательств": [
    {"описание": "срок действия доверенности", "срок": "1 год"}
  ]
}''',

# СЛОМАННЫЙ JSON: не закрыты массив и объект + OCR-ошибка (кириллическая О в КПП)
"doc_06_ocr_noise": '''{
  "номер_договора": "ПД-2026/08",
  "стороны": [
    {"роль": "Подрядчик", "название": "ООО «СтройМастер»", "ИНН": "6658101234", "КПП": "665801001"},
    {"роль": "Заказчик", "название": "АО «Девелопмент Урал»", "ИНН": "6634009988", "КПП": "6634010О1"}
  ],
  "даты": {"дата_подписания": "18.02.2026", "срок_действия": "до 18 августа 2026 года"},
  "суммы": [{"значение": 2800000, "валюта": "RUB", "описание": "стоимость работ, в т.ч. НДС"}],
  "сроки_обязательств": [
    {"описание": "аванс 30%", "срок": "7 рабочих дней"},
    {"описание": "выполнение работ", "срок": "60 календарных дней"}
''',

"doc_07_loan": '''{
  "номер_договора": "З-2026/45",
  "стороны": [
    {"роль": "Займодавец", "название": "ООО «Финанс Капитал»", "ИНН": "5260112233", "КПП": "526001001"},
    {"роль": "Заёмщик", "название": "ООО «Производство НН»", "ИНН": "5262556677", "КПП": "526201001"}
  ],
  "даты": {"дата_подписания": "10 июня 2026 г.", "срок_действия": "до полного исполнения обязательств"},
  "суммы": [{"значение": 5000000, "валюта": "RUB", "описание": "сумма займа под 12% годовых"}],
  "сроки_обязательств": [
    {"описание": "возврат займа и процентов", "срок": "до 10 декабря 2026 года"},
    {"описание": "уплата процентов", "срок": "ежемесячно не позднее 25-го числа"}
  ]
}''',
}

print("Загружено демо-ответов:", len(MOCK_RESPONSES))

Загружено демо-ответов: 7


## 8. Полный пайплайн NER

In [10]:
# ============================================================
#  ПОЛНЫЙ ПАЙПЛАЙН NER
# ============================================================
def get_raw_completion(doc: dict, temperature: float = 0.2) -> str:
    """Возвращает 'сырой' ответ модели: из mock-словаря или из реального API."""
    if USE_MOCK:
        return MOCK_RESPONSES.get(doc["id"], "{}")
    prepared = truncate(doc["text"])                 # контроль длины
    user_prompt = build_user_prompt(prepared)
    if USE_STRUCTURED_OUTPUT:                         # гарантированный JSON
        return call_yandexgpt_structured(SYSTEM_PROMPT, user_prompt,
                                         temperature=temperature)
    return call_yandexgpt_real(SYSTEM_PROMPT, user_prompt, temperature=temperature)


def run_ner(doc: dict, temperature: float = 0.2) -> dict:
    """Прогоняет один документ через NER-пайплайн.
    Возвращает словарь с сырым ответом, распарсенным JSON и статусом."""
    raw = get_raw_completion(doc, temperature=temperature)
    parsed, parse_note = parse_llm_json(raw)
    return {
        "id": doc["id"],
        "title": doc["title"],
        "raw": raw,
        "parsed": parsed,
        "ok": parsed is not None,
        "note": parse_note,
    }

## 9. Прогон всех документов: «фрагмент текста» → JSON

In [11]:
# ============================================================
#  ПРОГОН ВСЕХ ДОКУМЕНТОВ (5–7 шт.)
# ============================================================
import json

results = [run_ner(doc) for doc in DOCUMENTS]

for r in results:
    print("=" * 70)
    print(f"[{r['id']}]  {r['title']}")
    print(f"Парсинг: {'OK' if r['ok'] else 'ОШИБКА'}"
          + (f"  ({r['note']})" if r["note"] else ""))
    if r["parsed"] is not None:
        print(json.dumps(r["parsed"], ensure_ascii=False, indent=2))
    else:
        print("Сырой ответ модели:\n", r["raw"])

[doc_01_supply]  Договор поставки
Парсинг: OK
{
  "номер_договора": "ПС-2026/114",
  "стороны": [
    {
      "роль": "Поставщик",
      "название": "ООО «Ромашка»",
      "ИНН": "7701234567",
      "КПП": "770101001"
    },
    {
      "роль": "Покупатель",
      "название": "АО «Василёк»",
      "ИНН": "7809876543",
      "КПП": "780101001"
    }
  ],
  "даты": {
    "дата_подписания": "12 марта 2026 г.",
    "срок_действия": "до 31 декабря 2026 года"
  },
  "суммы": [
    {
      "значение": 1500000,
      "валюта": "RUB",
      "описание": "поставка офисной мебели, в т.ч. НДС 20%"
    }
  ],
  "сроки_обязательств": [
    {
      "описание": "поставка товара",
      "срок": "30 календарных дней"
    },
    {
      "описание": "оплата после получения товара",
      "срок": "10 рабочих дней"
    }
  ]
}
[doc_02_services]  Договор оказания услуг
Парсинг: OK
{
  "номер_договора": "У-77/2026",
  "стороны": [
    {
      "роль": "Исполнитель",
      "название": "ИП Сидоров Алексей Петрови

## 10. Edge case #1 — тест на отсутствие данных (галлюцинации)

Подаём документы без сумм (NDA, доверенность) и проверяем, что модель возвращает пустой
список `суммы`, а не выдумывает цифры.

In [12]:
# ============================================================
#  EDGE CASE #1 — ТЕСТ НА ОТСУТСТВИЕ ДАННЫХ (галлюцинации)
# ============================================================
# В NDA и доверенности НЕТ денежных сумм. Проверяем: вернула ли
# модель пустой список суммы (верно) или что-то выдумала (плохо).
# ------------------------------------------------------------
for r in results:
    if r["id"] in ("doc_04_nda", "doc_05_poa"):
        sums = (r["parsed"] or {}).get("суммы", None)
        verdict = "OK — суммы пустые, галлюцинаций нет" if sums == [] \
            else f"ВНИМАНИЕ — модель вернула суммы: {sums}"
        print(f"[{r['id']}] {r['title']}")
        print("  суммы =", sums, "->", verdict, "\n")

[doc_04_nda] NDA (без сумм — edge case)
  суммы = [] -> OK — суммы пустые, галлюцинаций нет 

[doc_05_poa] Доверенность (без сумм — edge case)
  суммы = [] -> OK — суммы пустые, галлюцинаций нет 



## 11. Edge case #2 — зашумлённый текст (OCR) и сломанный JSON

`doc_06` имитирует плохое распознавание скана; «модель» вернула JSON с незакрытыми скобками.
Смотрим, как парсер восстановил структуру и какие артефакты OCR остались в реквизитах.

In [13]:
# ============================================================
#  EDGE CASE #2 — ЗАШУМЛЕННЫЙ ТЕКСТ (плохой OCR) + СЛОМАННЫЙ JSON
# ============================================================
# doc_06 содержит опечатки распознавания (О вместо 0, латиница и т.п.),
# а «модель» вернула JSON с незакрытыми скобками. Смотрим, как
# парсер восстановил структуру и что осталось не так.
# ------------------------------------------------------------
import json

r = next(x for x in results if x["id"] == "doc_06_ocr_noise")
print(f"[{r['id']}] {r['title']}")
print("Статус парсинга:", "OK" if r["ok"] else "ОШИБКА")
print("Примечание парсера:", r["note"], "\n")

if r["parsed"]:
    print(json.dumps(r["parsed"], ensure_ascii=False, indent=2))
    # проверка ИНН/КПП на «нечисловые» символы (след OCR-ошибки)
    print("\nПроверка реквизитов на артефакты OCR:")
    for s in r["parsed"].get("стороны", []):
        for field in ("ИНН", "КПП"):
            val = s.get(field)
            if val and not val.isdigit():
                print(f"  ⚠ {s.get('роль')}: {field}='{val}' содержит нецифры "
                      f"(вероятно, O/О вместо 0)")

[doc_06_ocr_noise] Договор с OCR-опечатками (шум — edge case)


Статус парсинга: OK
Примечание парсера: JSON был повреждён и восстановлен (исходно: Expecting ',' delimiter: line 11 column 68 (char 595)) 

{
  "номер_договора": "ПД-2026/08",
  "стороны": [
    {
      "роль": "Подрядчик",
      "название": "ООО «СтройМастер»",
      "ИНН": "6658101234",
      "КПП": "665801001"
    },
    {
      "роль": "Заказчик",
      "название": "АО «Девелопмент Урал»",
      "ИНН": "6634009988",
      "КПП": "6634010О1"
    }
  ],
  "даты": {
    "дата_подписания": "18.02.2026",
    "срок_действия": "до 18 августа 2026 года"
  },
  "суммы": [
    {
      "значение": 2800000,
      "валюта": "RUB",
      "описание": "стоимость работ, в т.ч. НДС"
    }
  ],
  "сроки_обязательств": [
    {
      "описание": "аванс 30%",
      "срок": "7 рабочих дней"
    },
    {
      "описание": "выполнение работ",
      "срок": "60 календарных дней"
    }
  ]
}

Проверка реквизитов на артефакты OCR:
  ⚠ Заказчик: КПП='6634010О1' содержит нецифры (вероятно, O/О вместо 0)


## 12. Анализ узких мест (покрытие 4 типов сущностей)

In [14]:
# ============================================================
#  АНАЛИЗ УЗКИХ МЕСТ (сводная таблица покрытия сущностей)
# ============================================================
# Проверяем по каждому документу, извлечены ли все 4 типа сущностей,
# и не путает ли модель ИНН первой/второй стороны (уникальность ИНН).
# ------------------------------------------------------------
def coverage(parsed: dict) -> dict:
    parsed = parsed or {}
    parties = parsed.get("стороны", []) or []
    inns = [p.get("ИНН") for p in parties if p.get("ИНН")]
    return {
        "стороны+ИНН/КПП": bool(parties) and any(p.get("ИНН") for p in parties),
        "даты": bool((parsed.get("даты") or {}).get("дата_подписания")
                     or (parsed.get("даты") or {}).get("срок_действия")),
        "суммы": "суммы" in parsed,   # ключ присутствует (список может быть пуст)
        "сроки_обяз.": bool(parsed.get("сроки_обязательств")),
        "ИНН уникальны": len(inns) == len(set(inns)),
    }

cols = ["стороны+ИНН/КПП", "даты", "суммы", "сроки_обяз.", "ИНН уникальны"]
print(f"{'документ':<20}" + "".join(f"{c:<18}" for c in cols))
print("-" * 110)
for r in results:
    c = coverage(r["parsed"])
    row = f"{r['id']:<20}"
    for col in cols:
        row += f"{('✓' if c[col] else '✗'):<18}"
    print(row)

n_ok = sum(r["ok"] for r in results)
print(f"\nУспешно распарсено: {n_ok}/{len(results)} документов")

документ            стороны+ИНН/КПП   даты              суммы             сроки_обяз.       ИНН уникальны     


--------------------------------------------------------------------------------------------------------------
doc_01_supply       ✓                 ✓                 ✓                 ✓                 ✓                 
doc_02_services     ✓                 ✓                 ✓                 ✓                 ✓                 
doc_03_lease        ✓                 ✓                 ✓                 ✓                 ✓                 
doc_04_nda          ✓                 ✓                 ✓                 ✓                 ✓                 
doc_05_poa          ✓                 ✓                 ✓                 ✓                 ✓                 
doc_06_ocr_noise    ✓                 ✓                 ✓                 ✓                 ✓                 
doc_07_loan         ✓                 ✓                 ✓                 ✓                 ✓                 

Успешно распарсено: 7/7 документов


## 13. Итоговые промпты (для отчёта)

In [15]:
# ============================================================
#  ИТОГОВЫЕ ПРОМПТЫ (для отчёта)
# ============================================================
print("––– СИСТЕМНЫЙ ПРОМПТ –––\n")
print(SYSTEM_PROMPT)
print("\n\n––– ПОЛЬЗОВАТЕЛЬСКИЙ ПРОМПТ (пример для doc_01) –––\n")
print(build_user_prompt(DOCUMENTS[0]["text"]))

––– СИСТЕМНЫЙ ПРОМПТ –––

Ты — строгий и точный AI-помощник юриста. Твоя задача — извлекать факты из текста договоров без искажений и без домысливания. Если какого-то поля в тексте нет — верни для него null (для списков — пустой массив). Никогда не выдумывай значения. Отвечай ТОЛЬКО валидным JSON-объектом, без пояснений и без markdown.


––– ПОЛЬЗОВАТЕЛЬСКИЙ ПРОМПТ (пример для doc_01) –––

Извлеки из текста договора именованные сущности и верни JSON строго по следующей структуре:
{
  "номер_договора": "строка или null",
  "стороны": [
    {"роль": "строка (напр. Поставщик/Покупатель)", "название": "строка",
     "ИНН": "строка или null", "КПП": "строка или null"}
  ],
  "даты": {"дата_подписания": "строка или null", "срок_действия": "строка или null"},
  "суммы": [
    {"значение": "число", "валюта": "код валюты, напр. RUB", "описание": "за что"}
  ],
  "сроки_обязательств": [
    {"описание": "суть обязательства", "срок": "в днях/месяцах или дата"}
  ]
}

Извлекай ровно эти типы данны

## 14. Отчёт и выводы

### 14.1. Что настроено
- Аутентификация в Yandex Cloud через сервисный аккаунт и **API-ключ** (ключи не в коде).
- Подготовка текста: контроль длины, `truncate()` и `chunk_by_paragraphs()`.
- Система промптов: строгий **system prompt** (роль юриста, запрет домысливания) + **few-shot** пример.
- Интеграция по REST, `temperature=0.2`, `maxTokens=2000`.
- Устойчивый парсер: чистка markdown + починка сломанного JSON.
- Тесты на 7 документах, включая 3 граничных случая.

### 14.2. Примеры успешного извлечения
См. ячейку раздела 9 — для каждого документа выведено «фрагмент исходного текста → полученный JSON».
Все 4 типа сущностей извлекаются (раздел 12, сводная таблица покрытия).

### 14.3. Ошибки LLM и как их чинить
| Проблема | Где проявилась | Как решаем |
|---|---|---|
| Ответ обёрнут в ```` ```json ```` | `doc_01` | регуляркой снимаем markdown-ограждение перед `json.loads` |
| **Незакрытые скобки** в JSON | `doc_06` | функция `_try_repair()` добирает `}`/`]`, убирает висячие запятые |
| **OCR-артефакты** (кириллическая «О» вместо `0` в КПП) | `doc_06` | пост-валидация `.isdigit()` для ИНН/КПП, подсветка подозрительных значений |
| Риск **галлюцинации** суммы там, где её нет | NDA/доверенность | явный запрет в system prompt + `temperature≈0.2`; проверяем `суммы == []` |
| Путаница ИНН сторон | потенциально | проверка уникальности ИНН + роль стороны в схеме |

Дополнительно снижают ошибки: строгая JSON-схема в промпте, few-shot, а на новых версиях YandexGPT —
режим **структурированного вывода** (JSON Schema), который гарантирует валидный JSON на стороне API.

### 14.4. Вывод: готов ли YandexGPT к production-NER против классических моделей (Natasha, spaCy)?

**Плюсы LLM-подхода**
- Гибкость: новые типы сущностей добавляются правкой промпта, без переобучения и разметки.
- Понимает контекст и синонимы ролей («Поставщик»/«Продавец»), извлекает не только span, но и нормализует.
- Быстрый старт: рабочий прототип за часы.

**Минусы LLM-подхода**
- Недетерминированность и риск галлюцинаций; нужен контроль (`temperature`, валидация, тесты).
- JSON может «ломаться» — обязателен устойчивый парсер (реализован).
- Стоимость и задержка на объёмах; лимиты контекста → нужен chunking.
- Чувствительность к формулировкам промпта.

**Плюсы классики (Natasha/spaCy)**
- Быстро, дёшево, детерминированно, работает офлайн; идеально для стабильных типов (ФИО, даты, организации, деньги).

**Минусы классики**
- Требует разметки/правил под новые сущности; хуже с контекстом и нормализацией.

**Итог.** Для строго структурированных, высоконагруженных потоков однотипных документов надёжнее и дешевле
классические NER (Natasha/spaCy) или гибрид. Для гибкого извлечения по меняющимся требованиям, редких форматов
и быстрого прототипа — **YandexGPT применим в production при условии** обязательной обвязки: низкая температура,
строгая JSON-схема (лучше — structured output), валидация полей и авто-починка парсинга. Оптимум на практике —
**гибрид**: LLM извлекает и нормализует, классические правила/словари валидируют ИНН/КПП/даты.